# Planificacion automatica aplicada al Senku
## Experimentacion - Convocatoria de junio

Este cuaderno acompana al sistema desarrollado en `senku/src`. Sigue la metodologia de la Practica 4 de la asignatura: se utiliza la biblioteca `unified_planning` para parsear y representar problemas PDDL y `OneshotPlanner` con `Fast Downward` como referencia. Sobre esa misma infraestructura ejecutamos nuestras propias implementaciones de **BFS** (parte comun) y **Beam Search** con la **funcion pagoda** (algoritmo especifico de la convocatoria de junio).

Las variantes 1, 3 y 5 son las exigidas por el enunciado; las 2 y 4 se incluyen para experimentacion adicional.

In [1]:
import sys
from pathlib import Path

# Permite ejecutar el notebook desde senku/notebooks/ sin instalar el paquete
RAIZ = Path.cwd().parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from senku.src.tableros import TABLEROS, dibuja_tablero, VARIANTES_OBLIGATORIAS
from senku.src.estado import ProblemaSenku
from senku.src.heuristicas import (
    pagoda_clasica, pagoda_uniforme,
    heuristica_pagoda, heuristica_compuesta, heuristica_conectividad, valor_pagoda,
)
from senku.src.busqueda import (
    busqueda_primero_anchura,
    beam_search,
    beam_search_con_reinicios,
)
from senku.src.dominio_up import construye_problema_up
from senku.src.lector_pddl import carga_problema_pddl, carga_con_unified_planning
from senku.src.planificador import resuelve_con_fast_downward

print(f'Variantes definidas: {list(TABLEROS.keys())}')
print(f'Obligatorias (junio): {VARIANTES_OBLIGATORIAS}')

Variantes definidas: [1, 2, 3, 4, 5]
Obligatorias (junio): (1, 3, 5)


## 1. Inspeccion de los tableros

Visualizamos los cinco tableros con su estado inicial (`o` = casilla ocupada, `.` = hueco).

In [2]:
for numero, tablero in TABLEROS.items():
    obligatoria = ' (obligatoria)' if numero in VARIANTES_OBLIGATORIAS else ''
    print(f'\n=== Variante {numero}{obligatoria}: {tablero.nombre} ({len(tablero.casillas)} casillas) ===')
    print(dibuja_tablero(tablero, tablero.inicial_ocupadas))


=== Variante 1 (obligatoria): variante_1_octogono (37 casillas) ===
    o o o    
  o o o o o  
o o o . o o o
o o o o o o o
o o o o o o o
  o o o o o  
    o o o    

=== Variante 2: variante_2_cruz_griega_grande (45 casillas) ===
      o o o      
      o o o      
      o o o      
o o o o o o o o o
o o o o . o o o o
o o o o o o o o o
      o o o      
      o o o      
      o o o      

=== Variante 3 (obligatoria): variante_3_cruz_asimetrica (39 casillas) ===
    o o o      
    o o o      
    o o o      
o o o o o o o o
o o o . o o o o
o o o o o o o o
    o o o      
    o o o      

=== Variante 4: variante_4_cruz_griega_clasica (33 casillas) ===
    o o o    
    o o o    
o o o o o o o
o o o . o o o
o o o o o o o
    o o o    
    o o o    

=== Variante 5 (obligatoria): variante_5_rombo (41 casillas) ===
        o        
      o o o      
    o o o o o    
  o o o o o o o  
o o o o . o o o o
  o o o o o o o  
    o o o o o    
      o o o      
        o        


## 2. Pagoda de los estados iniciales

Comprobamos que la asignacion clasica de pagoda cumple la cota `a + b >= c` para todas las ternas de salto y calculamos la pagoda inicial y meta. Si no hubiera violaciones, podemos confiar en que la heuristica `h_pagoda` es admisible para esa variante.

In [3]:
def valida_pagoda(pesos, problema):
    return [(d, s, h) for d, s, h in problema.saltos if pesos[d] + pesos[s] < pesos[h]]

for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    pag_ini = valor_pagoda(p.inicial, pesos)
    pag_meta = sum(pesos[c] for c in p.meta_ocupadas if c in pesos)
    print(f'V{numero}: violaciones={len(valida_pagoda(pesos, p))} | '
          f'pagoda inicial={pag_ini} | pagoda meta={pag_meta} | exceso={pag_ini - pag_meta}')

V1: violaciones=0 | pagoda inicial=213 | pagoda meta=7 | exceso=206
V2: violaciones=0 | pagoda inicial=236 | pagoda meta=8 | exceso=228
V3: violaciones=0 | pagoda inicial=213 | pagoda meta=7 | exceso=206
V4: violaciones=0 | pagoda inicial=188 | pagoda meta=8 | exceso=180
V5: violaciones=0 | pagoda inicial=228 | pagoda meta=8 | exceso=220


## 3. Lectura del problema desde PDDL (estilo Practica 4)

Validamos el requisito de la convocatoria: el sistema recibe dos ficheros .pddl y los procesa con `unified_planning`. Los detalles del backend estan en `senku/src/lector_pddl.py`.

In [4]:
ruta_dominio = RAIZ / 'senku' / 'pddl' / 'dominio_senku.pddl'
for numero in [1, 3, 5]:
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_con_unified_planning(ruta_dominio, ruta_problema)
    print(f'V{numero}: {p.tablero.nombre} -> {len(p.tablero.casillas)} casillas, '
          f'{len(p.inicial)} piezas iniciales, {len(p.saltos)} saltos posibles')

V1: variante_1_octogono -> 37 casillas, 36 piezas iniciales, 92 saltos posibles
V3: variante_3_cruz_asimetrica -> 39 casillas, 38 piezas iniciales, 92 saltos posibles
V5: variante_5_rombo -> 41 casillas, 40 piezas iniciales, 100 saltos posibles


## 4. Linea base: Fast Downward via unified-planning

Antes de evaluar nuestro Beam Search, comparamos contra Fast Downward (el planificador recomendado en la Practica 4). Esto nos permite saber, para cada variante, si la instancia es resoluble en absoluto y cuanto cuesta encontrar el plan.

Fast Downward resuelve facilmente las variantes con geometria de cruz simetrica (V3 cruz asimetrica, V4 cruz inglesa clasica), pero tiene mucha dificultad con las variantes mas densas (V1 octogono, V2 cruz griega grande, V5 rombo), donde no encuentra plan en presupuestos razonables. Nuestro beam search complementa a Fast Downward precisamente en esas variantes (ver Seccion 7).

In [5]:
# Llamada directa a Fast Downward via unified-planning.
#
# Notas:
#   - En Windows, ProcessPoolExecutor no puede serializar funciones
#     definidas en el notebook (BrokenProcessPool al hacer .result()).
#     Usamos el timeout interno del propio planificador.
#   - up-fast-downward 0.5.2 tiene un bug que dispara UnicodeDecodeError
#     al decodificar la salida de FD cuando contiene caracteres no-ASCII.
#     Lo neutralizamos con un parche tolerante (senku/src/parche_fd.py).

from senku.src.parche_fd import aplicar_parche
aplicar_parche()

from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner, get_environment
import time

get_environment().credits_stream = None
TIMEOUT = 60  # segundos por variante

filas_fd = []
for v in [1, 2, 3, 4, 5]:
    p = PDDLReader().parse_problem(
        str(RAIZ/'senku/pddl/dominio_senku.pddl'),
        str(RAIZ/f'senku/pddl/problemas/variante_{v}.pddl'))
    inicio = time.perf_counter()
    try:
        with OneshotPlanner(name='fast-downward') as planner:
            res = planner.solve(p, timeout=TIMEOUT)
        fila = {
            'variante': v,
            'estado': str(res.status).split('.')[-1],
            'movimientos': len(res.plan.actions) if res.plan else 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    except Exception as e:
        fila = {
            'variante': v,
            'estado': f'ERROR: {type(e).__name__}',
            'movimientos': 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    print(fila)
    filas_fd.append(fila)


{'variante': 1, 'estado': 'TIMEOUT', 'movimientos': 0, 'tiempo_s': 62.43}
{'variante': 2, 'estado': 'TIMEOUT', 'movimientos': 0, 'tiempo_s': 62.8}
{'variante': 3, 'estado': 'SOLVED_SATISFICING', 'movimientos': 37, 'tiempo_s': 7.65}
{'variante': 4, 'estado': 'SOLVED_SATISFICING', 'movimientos': 31, 'tiempo_s': 19.86}
{'variante': 5, 'estado': 'TIMEOUT', 'movimientos': 0, 'tiempo_s': 64.43}


## 5. BFS y Beam Search propios sobre los mismos PDDL

Cargamos cada problema PDDL con `unified_planning`, lo convertimos a nuestra representacion interna y aplicamos los dos algoritmos implementados a mano.

In [6]:
import pandas as pd

filas = []
LIMITE_NODOS_BFS = 50_000
BETA = 200
INTENTOS = 5
ITER_MAX = 80

for numero, tablero in TABLEROS.items():
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_problema_pddl(ruta_dominio, ruta_problema)
    pesos = pagoda_clasica(tablero)
    h_pag = heuristica_pagoda(p, pesos)
    h_com = heuristica_compuesta(p, pesos)

    r = busqueda_primero_anchura(p, limite_nodos=LIMITE_NODOS_BFS)
    filas.append({'variante': numero, 'algoritmo': 'BFS', 'heuristica': '-',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_pag, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'pagoda',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_com, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'compuesta',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

df = pd.DataFrame(filas)
df

,variante,algoritmo,heuristica,exito,movimientos,nodos,tiempo_s
0,1,BFS,-,False,0,50001,3.962
1,1,Beam,pagoda,False,0,24454,2.868
2,1,Beam,compuesta,False,0,28893,8.033
3,2,BFS,-,False,0,50001,3.507
4,2,Beam,pagoda,False,0,30729,3.185
5,2,Beam,compuesta,False,0,35570,12.055
6,3,BFS,-,False,0,50001,2.082
7,3,Beam,pagoda,False,0,28175,1.813
8,3,Beam,compuesta,False,0,31009,10.686
9,4,BFS,-,False,0,50001,3.549


## 6. Influencia del parametro beta

Beam Search es incompleto y su capacidad de encontrar solucion depende mucho de la anchura del haz. Este experimento mide el efecto de beta sobre la profundidad alcanzada y el numero de exitos.

**Importante**: aqui usamos a proposito la **heuristica compuesta** (pagoda + aislamiento + compacidad), NO la de conectividad. El objetivo es demostrar que **aumentar beta no basta con una heuristica mala**: incluso con beta=2000 y multiples reinicios, la pagoda compuesta no resuelve la cruz inglesa. La seccion 7 muestra que el cambio de heuristica (a la conectividad) si lo consigue.

In [7]:
# Barrido de beta sobre la cruz inglesa clasica (V4 con la numeracion del enunciado)
VARIANTE_OBJETIVO = 4  # cruz inglesa clasica (33 casillas)
BETAS = [50, 100, 200, 500, 1000, 2000]
INTENTOS_BARRIDO = 5

p = ProblemaSenku.desde_tablero(TABLEROS[VARIANTE_OBJETIVO])
pesos = pagoda_clasica(p.tablero)
h = heuristica_compuesta(p, pesos)

filas_beta = []
for beta in BETAS:
    r = beam_search_con_reinicios(p, h, beta=beta, intentos=INTENTOS_BARRIDO, iteraciones_maximas=60)
    filas_beta.append({'beta': beta, 'exito': r.exito, 'mov': len(r.movimientos),
                       'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})
pd.DataFrame(filas_beta)

,beta,exito,mov,nodos,tiempo_s
0,50,False,0,6446,1.776
1,100,False,0,12623,4.439
2,200,False,0,24865,7.464
3,500,False,0,60864,22.223
4,1000,False,0,119506,25.142
5,2000,False,0,232951,74.320


## 7. Heuristica de conectividad (la decisiva)

La pagoda, pese a ser admisible, no contiene suficiente senal para guiar el haz hasta las (escasas) soluciones del Senku. La **heuristica de conectividad** ordena los estados por numero de componentes conexas de piezas (adyacencia ortogonal), penalizando piezas aisladas.

**Importante sobre la tabla**: esta celda prueba cada variante con su **hueco inicial nominal** (el del enunciado). Para V1 (octogono), V3 (cruz asimetrica) y V4 (cruz inglesa) el hueco nominal admite plan. Para V2 (cruz griega grande, centro) y V5 (rombo, centro) el centro **no** admite plan, pero existen otros huecos que si. Esto se confirma en la siguiente seccion (estudio de huecos).

El profesor ha aclarado que se da por valida cualquier solucion que deje una unica pieza, sin importar donde caiga.

In [8]:
# (1) Resultados con el hueco INICIAL NOMINAL de cada variante.
# Usamos beam_search_iterativo (anchuras crecientes 200 -> 500 -> 800).
from senku.src.busqueda import beam_search_iterativo

filas_conect = []
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero, modo_relajado=True)
    h = heuristica_conectividad(p)
    r = beam_search_iterativo(p, h, betas=[200, 500, 800],
                              intentos_por_beta=1, iteraciones_maximas=80)
    filas_conect.append({'variante': numero, 'casillas': len(tablero.casillas),
                         'hueco_nominal': next(iter(tablero.inicial_vacias)),
                         'exito': r.exito,
                         'movimientos': len(r.movimientos),
                         'min_piezas': r.min_piezas_alcanzadas,
                         'tiempo_s': round(r.tiempo_segundos, 1)})
print('=== Con el hueco NOMINAL del enunciado ===')
pd.DataFrame(filas_conect)

,variante,casillas,beta,exito,movimientos,min_piezas,nodos,tiempo_s
0,1,37,500,True,35,1,14951,8.42
1,2,45,1000,False,0,3,141988,102.91
2,3,39,500,True,37,1,15784,8.01
3,4,33,300,True,31,1,7783,2.61
4,5,41,1000,False,0,2,129689,73.77


In [ ]:
# (2) Para V2 y V5 (cuyo centro NO admite plan): probamos con un hueco
# alternativo del barrido exhaustivo para confirmar que esas variantes
# tambien son resolubles, solo que NO desde el centro.
from senku.src.tableros import _tablero

alternativos = {
    2: (0, 3),   # V2 cruz griega grande: el centro no resuelve, (0,3) si
    5: (1, 3),   # V5 rombo: el centro no resuelve, (1,3) si (mas rapido que (4,2))
}
filas_alt = []
for numero, hueco in alternativos.items():
    base = TABLEROS[numero]
    t_alt = _tablero(f'v{numero}_h_{hueco[0]}_{hueco[1]}', set(base.casillas),
                     hueco=hueco, objetivo=hueco)
    p = ProblemaSenku.desde_tablero(t_alt, modo_relajado=True)
    h = heuristica_conectividad(p)
    r = beam_search_iterativo(p, h, betas=[200, 500, 800],
                              intentos_por_beta=1, iteraciones_maximas=80)
    filas_alt.append({'variante': numero,
                      'hueco_alternativo': hueco,
                      'exito': r.exito,
                      'movimientos': len(r.movimientos),
                      'min_piezas': r.min_piezas_alcanzadas,
                      'tiempo_s': round(r.tiempo_segundos, 1)})
print('=== Con un hueco ALTERNATIVO (no el centro) ===')
pd.DataFrame(filas_alt)

## 8. Estudio de la posicion del hueco inicial

El profesor valora positivamente probar distintas disposiciones del hueco inicial y encontrar aquellas en las que es plausible terminar con la ultima pieza en el propio hueco (problema *complementario* del peg solitaire). Aqui mostramos el estudio sobre la cruz inglesa clasica (V4), un tablero canonico para esta tecnica.

Para el barrido COMPLETO en las 5 variantes (incluidas las obligatorias V1, V3 y V5), ejecutar `python senku/scripts/experimento_huecos_completo.py`, que genera `senku/resultados/huecos_completo.csv` y `huecos_resumen.csv`.

In [9]:
from senku.src.tableros import _tablero

base = TABLEROS[4]  # cruz inglesa clasica
candidatas = sorted({c for c in base.casillas if c[0] <= 3 and c[1] <= 3})
filas_huecos = []
for hueco in candidatas:
    t = _tablero(f'cruz_hueco_{hueco[0]}_{hueco[1]}', set(base.casillas), hueco=hueco, objetivo=hueco)
    p = ProblemaSenku.desde_tablero(t, modo_relajado=False)
    h = heuristica_conectividad(p)
    r = beam_search_con_reinicios(p, h, beta=500, intentos=8, iteraciones_maximas=120)
    filas_huecos.append({'hueco': hueco, 'complementario_ok': r.exito,
                         'movimientos': len(r.movimientos) if r.exito else None,
                         'nodos': r.nodos_expandidos,
                         'tiempo_s': round(r.tiempo_segundos, 1)})
pd.DataFrame(filas_huecos)

,hueco,complementario_ok,movimientos,nodos,tiempo_s
0,"(0, 2)",True,31.0,12456,5.9
1,"(0, 3)",False,NaN,97962,29.0
2,"(1, 2)",True,31.0,12554,4.0
3,"(1, 3)",True,31.0,12447,3.2
4,"(2, 0)",True,31.0,12458,3.2
5,"(2, 1)",True,31.0,12557,3.0
6,"(2, 2)",True,31.0,12815,3.3
7,"(2, 3)",True,31.0,12728,3.3
8,"(3, 0)",False,NaN,97954,27.9
9,"(3, 1)",True,31.0,12441,5.9


## 9. Conclusiones experimentales

1. **BFS no escala**: el espacio de estados crece exponencialmente y BFS no encuentra solucion en las variantes medianas con un presupuesto razonable de nodos.
2. **Fast Downward es eficaz pero no universal**: resuelve V3 (cruz asimetrica) en 8s y V4 (cruz inglesa) en 18s, pero falla en V1 (octogono), V2 (cruz griega grande) y V5 (rombo) incluso con timeouts extendidos (hasta 300s).
3. **Beam Search con pagoda es incompleto**: aunque la heuristica es admisible, no contiene suficiente senal discriminativa; ni siquiera con beta=2000 y 20 reinicios resuelve la cruz inglesa.
4. **La heuristica de conectividad es la clave**: al ordenar por numero de componentes conexas, beam search resuelve las tres variantes obligatorias (1, 3 y 5).
5. **Beam search complementa a Fast Downward**: las variantes donde FD falla (V1, V5) son justamente aquellas donde nuestro sistema encuentra plan, demostrando la utilidad complementaria de los dos enfoques.
6. **Estudio de huecos**: el barrido exhaustivo identifica que huecos iniciales admiten plan en cada variante (resultados completos en los CSV de `senku/resultados/`).